
# 2026 Political Events × Agricultural ETF Event Study

Starter notebook for a weekly article series studying how agricultural commodity-linked ETFs behave around major political events.

**Trade ETFs**: `DBA`, `WEAT`, `CORN`, `SOYB`, `CANE`  
**Context ETFs**: `DBC`, `UUP`, `USO`

The notebook includes event-window returns, abnormal returns, context comparison, Brownian-motion/GBM Monte Carlo, block-bootstrap Monte Carlo, and article-ready summaries.

This is exploratory research, not investment advice.


In [ ]:

# Optional install if needed:
# !pip install yfinance pandas numpy matplotlib

from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Iterable, Optional

try:
    import yfinance as yf
except ImportError as exc:
    raise ImportError("Install yfinance first: pip install yfinance") from exc

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


In [ ]:

TRADE_TICKERS = ("DBA", "WEAT", "CORN", "SOYB", "CANE")
CONTEXT_TICKERS = ("DBC", "UUP", "USO")
ALL_TICKERS = list(TRADE_TICKERS + CONTEXT_TICKERS)

START_DATE = "2025-01-01"
END_DATE = None

PRE_EVENT_DAYS = 10
POST_EVENT_DAYS = 20

ESTIMATION_WINDOW_DAYS = 60
ESTIMATION_GAP_DAYS = 5

MC_HORIZON_DAYS = 20
MC_N_PATHS = 5000


## 2026 political event calendar

This is a living starter calendar. Add more events weekly as new policy, trade, climate, election, or geopolitical items appear.

In [ ]:
events = [{'event_id': 'FRA_MUNICIPAL_R1_2026', 'event_date': '2026-03-15', 'event_name': 'France municipal elections — first round', 'country_region': 'France / EU', 'category': 'Election', 'agri_relevance': 'Moderate: EU political barometer; possible policy, EUR/USD, and broader risk-sentiment read-through.', 'source_name': 'Reuters', 'source_url': 'https://www.reuters.com/world/what-you-need-know-about-frances-local-elections-2026-03-05/', 'include': True}, {'event_id': 'FRA_MUNICIPAL_R2_2026', 'event_date': '2026-03-22', 'event_name': 'France municipal elections — second round', 'country_region': 'France / EU', 'category': 'Election', 'agri_relevance': 'Moderate: local election as national political signal ahead of 2027; possible EU policy/risk read-through.', 'source_name': 'Reuters', 'source_url': 'https://www.reuters.com/world/what-you-need-know-about-frances-local-elections-2026-03-05/', 'include': True}, {'event_id': 'PERU_GENERAL_2026', 'event_date': '2026-04-12', 'event_name': 'Peru general election', 'country_region': 'Peru', 'category': 'Election', 'agri_relevance': 'Low-to-moderate: Latin America political-risk marker; likely regional sentiment more than direct agri ETF channel.', 'source_name': 'ONPE', 'source_url': 'https://www.gob.pe/institucion/onpe/campa%C3%B1as/104066-elecciones-generales-2026', 'include': True}, {'event_id': 'COL_PRES_R1_2026', 'event_date': '2026-05-31', 'event_name': 'Colombia presidential election — first round', 'country_region': 'Colombia', 'category': 'Election', 'agri_relevance': 'Moderate: Colombia is relevant to soft-commodity/political-risk monitoring, though not directly mapped to the listed ETFs.', 'source_name': 'Registraduría Nacional', 'source_url': 'https://www.registraduria.gov.co/Elecciones-2026', 'include': True}, {'event_id': 'COL_PRES_R2_2026', 'event_date': '2026-06-21', 'event_name': 'Colombia presidential election — second round', 'country_region': 'Colombia', 'category': 'Election', 'agri_relevance': 'Moderate: political outcome may influence regional risk sentiment.', 'source_name': 'Registraduría Nacional', 'source_url': 'https://www.registraduria.gov.co/Elecciones-2026', 'include': True}, {'event_id': 'BRAZIL_GENERAL_R1_2026', 'event_date': '2026-10-04', 'event_name': 'Brazil general election — first round', 'country_region': 'Brazil', 'category': 'Election', 'agri_relevance': 'High: Brazil is central to soybeans, corn, sugar, and global agri trade. Watch SOYB, CORN, CANE, DBA.', 'source_name': 'Tribunal Superior Eleitoral', 'source_url': 'https://www.tse.jus.br/legislacao/compilada/res/2026/resolucao-no-23-751-de-26-de-fevereiro-de-2026', 'include': True}, {'event_id': 'BRAZIL_GENERAL_R2_2026', 'event_date': '2026-10-25', 'event_name': 'Brazil general election — second round if required', 'country_region': 'Brazil', 'category': 'Election', 'agri_relevance': 'High: second-round result could matter for agri export policy, currency expectations, and soft/grain markets.', 'source_name': 'Tribunal Superior Eleitoral', 'source_url': 'https://www.tse.jus.br/legislacao/compilada/res/2026/resolucao-no-23-751-de-26-de-fevereiro-de-2026', 'include': True}, {'event_id': 'US_MIDTERMS_2026', 'event_date': '2026-11-03', 'event_name': 'United States congressional general election / midterms', 'country_region': 'United States', 'category': 'Election', 'agri_relevance': 'High: US trade policy, ethanol/biofuels, farm policy, fiscal policy, USD, and China trade expectations can affect agri ETFs.', 'source_name': 'FEC', 'source_url': 'https://www.fec.gov/help-candidates-and-committees/dates-and-deadlines/2026-reporting-dates/electioneering-communications-periods-congressional-primaries-2026/', 'include': True}, {'event_id': 'COP31_2026', 'event_date': '2026-11-09', 'event_name': 'UN Climate Change Conference COP31 — start', 'country_region': 'Antalya, Türkiye / Global', 'category': 'Climate policy', 'agri_relevance': 'Moderate-to-high: climate policy, land use, food security, biofuels, and carbon policy can influence agriculture narratives.', 'source_name': 'UNFCCC', 'source_url': 'https://unfccc.int/cop31', 'include': True}, {'event_id': 'G20_MIAMI_2026', 'event_date': '2026-12-14', 'event_name': "G20 Leaders' Summit — start", 'country_region': 'United States / Global', 'category': 'Summit', 'agri_relevance': 'High: global trade, energy, regulation, and supply-chain policy agenda. Watch DBC, UUP, USO context around agri ETFs.', 'source_name': 'US Department of State', 'source_url': 'https://www.state.gov/releases/office-of-the-spokesperson/2025/12/united-states-hosts-first-g20-sherpa-meeting/', 'include': True}]

events_df = pd.DataFrame(events)
events_df["event_date"] = pd.to_datetime(events_df["event_date"])
events_df = events_df[events_df["include"]].sort_values("event_date").reset_index(drop=True)
events_df


## Download ETF data

In [ ]:

def download_ohlcv(tickers: Iterable[str], start: str, end: Optional[str] = None) -> pd.DataFrame:
    raw = yf.download(
        tickers=list(tickers),
        start=start,
        end=end,
        auto_adjust=True,
        group_by="ticker",
        progress=False,
        threads=True,
    )

    frames = []
    if isinstance(raw.columns, pd.MultiIndex):
        for ticker in tickers:
            if ticker not in raw.columns.get_level_values(0):
                print(f"Warning: {ticker} not found in download.")
                continue
            tmp = raw[ticker].copy()
            tmp["ticker"] = ticker
            frames.append(tmp.reset_index())
    else:
        tmp = raw.copy()
        tmp["ticker"] = list(tickers)[0]
        frames.append(tmp.reset_index())

    df = pd.concat(frames, ignore_index=True)
    df.columns = [str(c).lower().replace(" ", "_") for c in df.columns]
    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
    return df

prices = download_ohlcv(ALL_TICKERS, START_DATE, END_DATE)
print(prices.shape)
prices.tail()


## Feature engineering and return matrices

In [ ]:

def add_return_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values(["ticker", "date"])
    g = df.groupby("ticker", group_keys=False)

    df["ret_1d"] = g["close"].pct_change()
    df["log_ret_1d"] = np.log(df["close"]).groupby(df["ticker"]).diff()
    df["range_pct"] = (df["high"] - df["low"]) / df["close"].replace(0, np.nan)
    df["volume_chg"] = g["volume"].pct_change()

    for w in [5, 10, 20, 60]:
        df[f"vol_{w}d"] = g["log_ret_1d"].transform(lambda s: s.rolling(w, min_periods=max(5, w // 2)).std())
        df[f"ret_{w}d"] = g["close"].transform(lambda s: s.pct_change(w))
        df[f"ma_{w}d"] = g["close"].transform(lambda s: s.rolling(w, min_periods=max(5, w // 2)).mean())

    df["ma_gap_20"] = df["close"] / df["ma_20d"] - 1
    df["ma_gap_60"] = df["close"] / df["ma_60d"] - 1
    df["range_shock_20"] = df["range_pct"] / g["range_pct"].transform(lambda s: s.shift(1).rolling(20, min_periods=10).median())
    return df

features = add_return_features(prices)
close_wide = features.pivot(index="date", columns="ticker", values="close").sort_index()
ret_wide = close_wide.pct_change()
logret_wide = np.log(close_wide).diff()

print("Available date range:", close_wide.index.min().date(), "to", close_wide.index.max().date())
close_wide.tail()


## Event-study helpers

In [ ]:

def nearest_trading_day(date: pd.Timestamp, trading_index: pd.DatetimeIndex, method: str = "next") -> pd.Timestamp:
    date = pd.Timestamp(date).normalize()
    idx = trading_index.sort_values()

    if date in idx:
        return date

    if method == "next":
        future = idx[idx >= date]
        return future[0] if len(future) else idx[-1]

    if method == "previous":
        past = idx[idx <= date]
        return past[-1] if len(past) else idx[0]

    pos = np.argmin(np.abs(idx - date))
    return idx[pos]


def event_window_returns(
    returns_wide: pd.DataFrame,
    event_date: pd.Timestamp,
    pre_days: int = 10,
    post_days: int = 20,
    tickers: Optional[Iterable[str]] = None,
) -> pd.DataFrame:
    tickers = list(tickers or returns_wide.columns)
    trading_date = nearest_trading_day(event_date, returns_wide.index, method="next")
    loc = returns_wide.index.get_loc(trading_date)

    start = max(0, loc - pre_days)
    end = min(len(returns_wide) - 1, loc + post_days)

    window = returns_wide.iloc[start:end + 1][tickers].copy()
    window["event_trading_date"] = trading_date
    window["relative_day"] = np.arange(start - loc, end - loc + 1)
    window = window.reset_index()

    long = window.melt(
        id_vars=["date", "event_trading_date", "relative_day"],
        value_vars=tickers,
        var_name="ticker",
        value_name="daily_return",
    )
    return long


def build_all_event_windows(
    events_df: pd.DataFrame,
    returns_wide: pd.DataFrame,
    tickers: Iterable[str],
    pre_days: int = 10,
    post_days: int = 20,
) -> pd.DataFrame:
    frames = []
    for _, ev in events_df.iterrows():
        win = event_window_returns(returns_wide, ev["event_date"], pre_days, post_days, tickers)
        for col in ["event_id", "event_name", "country_region", "category", "agri_relevance"]:
            win[col] = ev[col]
        win["event_calendar_date"] = ev["event_date"]
        frames.append(win)
    return pd.concat(frames, ignore_index=True)


event_windows = build_all_event_windows(events_df, ret_wide, ALL_TICKERS, PRE_EVENT_DAYS, POST_EVENT_DAYS)

event_windows["cum_return"] = (
    event_windows.sort_values(["event_id", "ticker", "relative_day"])
    .groupby(["event_id", "ticker"])["daily_return"]
    .transform(lambda s: (1 + s.fillna(0)).cumprod() - 1)
)

event_windows.head()


## Event-window summary

In [ ]:

summary = (
    event_windows[event_windows["relative_day"].between(0, POST_EVENT_DAYS)]
    .groupby(["event_id", "event_name", "ticker"], as_index=False)
    .agg(
        event_start=("event_calendar_date", "first"),
        event_trading_date=("event_trading_date", "first"),
        cum_return_0_to_post=("daily_return", lambda s: (1 + s.fillna(0)).prod() - 1),
        avg_daily_return=("daily_return", "mean"),
        realized_vol=("daily_return", "std"),
        positive_days=("daily_return", lambda s: (s > 0).mean()),
    )
)

summary_trade = summary[summary["ticker"].isin(TRADE_TICKERS)].sort_values(["event_start", "cum_return_0_to_post"])
summary_trade


In [ ]:

def plot_event_cumulative(event_id: str, tickers: Iterable[str] = TRADE_TICKERS, include_context: bool = True):
    tickers = list(tickers)
    if include_context:
        tickers = tickers + list(CONTEXT_TICKERS)

    df = event_windows[(event_windows["event_id"] == event_id) & (event_windows["ticker"].isin(tickers))].copy()
    if df.empty:
        print(f"No data for event_id={event_id}")
        return

    event_name = df["event_name"].iloc[0]
    event_trading_date = pd.Timestamp(df["event_trading_date"].iloc[0]).date()

    plt.figure(figsize=(12, 6))
    for ticker, g in df.groupby("ticker"):
        g = g.sort_values("relative_day")
        plt.plot(g["relative_day"], g["cum_return"], label=ticker)

    plt.axvline(0, linestyle="--", linewidth=1)
    plt.axhline(0, linewidth=1)
    plt.title(f"{event_name} | trading date {event_trading_date}")
    plt.xlabel("Relative trading day")
    plt.ylabel("Cumulative return")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_event_cumulative(events_df["event_id"].iloc[0])


## Simple abnormal returns

Expected return is estimated from a pre-event window, not a full causal model.

In [ ]:

def estimate_pre_event_params(
    returns_wide: pd.DataFrame,
    event_date: pd.Timestamp,
    tickers: Iterable[str],
    estimation_window: int = 60,
    estimation_gap: int = 5,
) -> pd.DataFrame:
    trading_date = nearest_trading_day(event_date, returns_wide.index, method="next")
    loc = returns_wide.index.get_loc(trading_date)
    start = max(0, loc - estimation_gap - estimation_window)
    end = max(0, loc - estimation_gap)
    hist = returns_wide.iloc[start:end][list(tickers)]

    rows = []
    for ticker in tickers:
        s = hist[ticker].dropna()
        rows.append({
            "ticker": ticker,
            "event_trading_date": trading_date,
            "mu_daily": s.mean(),
            "sigma_daily": s.std(),
            "n_estimation_days": len(s),
        })
    return pd.DataFrame(rows)


def abnormal_event_summary(events_df, returns_wide, tickers, post_days=20, estimation_window=60, estimation_gap=5):
    rows = []
    for _, ev in events_df.iterrows():
        params = estimate_pre_event_params(returns_wide, ev["event_date"], tickers, estimation_window, estimation_gap)
        win = event_window_returns(returns_wide, ev["event_date"], pre_days=0, post_days=post_days, tickers=tickers)
        actual = (
            win.groupby("ticker")["daily_return"]
            .apply(lambda s: (1 + s.fillna(0)).prod() - 1)
            .reset_index(name="actual_cum_return")
        )
        merged = actual.merge(params, on="ticker", how="left")
        merged["expected_cum_return"] = (1 + merged["mu_daily"]).pow(post_days + 1) - 1
        merged["abnormal_cum_return"] = merged["actual_cum_return"] - merged["expected_cum_return"]
        merged["event_id"] = ev["event_id"]
        merged["event_name"] = ev["event_name"]
        merged["event_calendar_date"] = ev["event_date"]
        merged["category"] = ev["category"]
        rows.append(merged)
    return pd.concat(rows, ignore_index=True)


abnormal_summary = abnormal_event_summary(
    events_df, ret_wide, TRADE_TICKERS, POST_EVENT_DAYS, ESTIMATION_WINDOW_DAYS, ESTIMATION_GAP_DAYS
)

abnormal_summary.sort_values(["event_calendar_date", "abnormal_cum_return"]).head(20)


## Context correlation around event windows

In [ ]:

def event_context_correlation(event_id: str, trade_tickers: Iterable[str] = TRADE_TICKERS, context_tickers: Iterable[str] = CONTEXT_TICKERS):
    df = event_windows[event_windows["event_id"] == event_id].copy()
    wide = df.pivot(index="relative_day", columns="ticker", values="daily_return")
    rows = []
    for t in trade_tickers:
        for c in context_tickers:
            if t in wide.columns and c in wide.columns:
                rows.append({"event_id": event_id, "trade_ticker": t, "context_ticker": c, "corr_window": wide[t].corr(wide[c])})
    return pd.DataFrame(rows)

all_context_corr = pd.concat([event_context_correlation(eid) for eid in events_df["event_id"]], ignore_index=True)
all_context_corr.head(20)



## Brownian-motion / GBM Monte Carlo

This compares actual post-event price paths with simulated paths estimated from pre-event daily log returns.

Interpretation:

- actual above 95% band → stronger than recent volatility suggested;
- actual below 5% band → weaker than recent volatility suggested;
- inside band → not unusual under the local pre-event volatility benchmark.


In [ ]:

def gbm_simulation_from_pre_event(
    close_wide: pd.DataFrame,
    logret_wide: pd.DataFrame,
    ticker: str,
    event_date: pd.Timestamp,
    estimation_window: int = 60,
    estimation_gap: int = 5,
    horizon_days: int = 20,
    n_paths: int = 5000,
    random_seed: int = 42,
) -> pd.DataFrame:
    local_rng = np.random.default_rng(random_seed)
    trading_date = nearest_trading_day(event_date, close_wide.index, method="next")
    loc = close_wide.index.get_loc(trading_date)
    start = max(0, loc - estimation_gap - estimation_window)
    end = max(0, loc - estimation_gap)
    hist = logret_wide.iloc[start:end][ticker].dropna()

    mu = hist.mean()
    sigma = hist.std()
    s0 = close_wide.loc[trading_date, ticker]

    shocks = local_rng.normal(loc=(mu - 0.5 * sigma**2), scale=sigma, size=(n_paths, horizon_days))
    log_paths = np.cumsum(shocks, axis=1)
    price_paths = s0 * np.exp(log_paths)
    price_paths = np.concatenate([np.full((n_paths, 1), s0), price_paths], axis=1)

    q = np.quantile(price_paths, [0.05, 0.25, 0.50, 0.75, 0.95], axis=0)
    return pd.DataFrame({
        "relative_day": np.arange(horizon_days + 1),
        "p05": q[0], "p25": q[1], "p50": q[2], "p75": q[3], "p95": q[4],
        "event_trading_date": trading_date, "ticker": ticker,
        "mu_daily_log": mu, "sigma_daily_log": sigma, "s0": s0,
    })


def block_bootstrap_simulation_from_pre_event(
    close_wide: pd.DataFrame,
    logret_wide: pd.DataFrame,
    ticker: str,
    event_date: pd.Timestamp,
    estimation_window: int = 120,
    estimation_gap: int = 5,
    horizon_days: int = 20,
    n_paths: int = 5000,
    block_size: int = 5,
    random_seed: int = 42,
) -> pd.DataFrame:
    local_rng = np.random.default_rng(random_seed)
    trading_date = nearest_trading_day(event_date, close_wide.index, method="next")
    loc = close_wide.index.get_loc(trading_date)
    start = max(0, loc - estimation_gap - estimation_window)
    end = max(0, loc - estimation_gap)
    hist = logret_wide.iloc[start:end][ticker].dropna().values
    s0 = close_wide.loc[trading_date, ticker]

    if len(hist) < block_size + 1:
        raise ValueError(f"Not enough historical returns for {ticker} before {event_date}")

    paths = []
    for _ in range(n_paths):
        sampled = []
        while len(sampled) < horizon_days:
            i = local_rng.integers(0, len(hist) - block_size + 1)
            sampled.extend(hist[i:i + block_size])
        sampled = np.array(sampled[:horizon_days])
        price_path = s0 * np.exp(np.cumsum(sampled))
        paths.append(np.concatenate([[s0], price_path]))

    paths = np.vstack(paths)
    q = np.quantile(paths, [0.05, 0.25, 0.50, 0.75, 0.95], axis=0)
    return pd.DataFrame({
        "relative_day": np.arange(horizon_days + 1),
        "p05": q[0], "p25": q[1], "p50": q[2], "p75": q[3], "p95": q[4],
        "event_trading_date": trading_date, "ticker": ticker, "s0": s0,
    })


def actual_price_path(close_wide, ticker, event_date, horizon_days=20):
    trading_date = nearest_trading_day(event_date, close_wide.index, method="next")
    loc = close_wide.index.get_loc(trading_date)
    end = min(len(close_wide) - 1, loc + horizon_days)
    s = close_wide.iloc[loc:end + 1][ticker].dropna()
    return pd.DataFrame({
        "relative_day": np.arange(len(s)),
        "actual_price": s.values,
        "calendar_date": s.index,
        "ticker": ticker,
        "event_trading_date": trading_date,
    })


def plot_monte_carlo_event(event_id: str, ticker: str, method: str = "gbm", horizon_days: int = MC_HORIZON_DAYS):
    ev = events_df.loc[events_df["event_id"] == event_id].iloc[0]

    if method == "gbm":
        sim = gbm_simulation_from_pre_event(
            close_wide, logret_wide, ticker, ev["event_date"],
            ESTIMATION_WINDOW_DAYS, ESTIMATION_GAP_DAYS, horizon_days, MC_N_PATHS, RANDOM_SEED
        )
    elif method == "bootstrap":
        sim = block_bootstrap_simulation_from_pre_event(
            close_wide, logret_wide, ticker, ev["event_date"],
            estimation_window=max(120, ESTIMATION_WINDOW_DAYS), estimation_gap=ESTIMATION_GAP_DAYS,
            horizon_days=horizon_days, n_paths=MC_N_PATHS, block_size=5, random_seed=RANDOM_SEED
        )
    else:
        raise ValueError("method must be 'gbm' or 'bootstrap'")

    actual = actual_price_path(close_wide, ticker, ev["event_date"], horizon_days)

    plt.figure(figsize=(11, 6))
    plt.fill_between(sim["relative_day"], sim["p05"], sim["p95"], alpha=0.2, label="5–95% scenario band")
    plt.fill_between(sim["relative_day"], sim["p25"], sim["p75"], alpha=0.3, label="25–75% scenario band")
    plt.plot(sim["relative_day"], sim["p50"], linestyle="--", label="median scenario")
    plt.plot(actual["relative_day"], actual["actual_price"], linewidth=2, label="actual")
    plt.title(f"{ticker} around {ev['event_name']} | {method.upper()} scenario benchmark")
    plt.xlabel("Trading days after event")
    plt.ylabel("Price")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    return sim, actual


# Examples:
# plot_monte_carlo_event("BRAZIL_GENERAL_R1_2026", "SOYB", method="gbm")
# plot_monte_carlo_event("BRAZIL_GENERAL_R1_2026", "SOYB", method="bootstrap")


## Monte Carlo diagnostic table

In [ ]:

def mc_terminal_diagnostic(event_id: str, ticker: str, method: str = "gbm", horizon_days: int = MC_HORIZON_DAYS) -> dict:
    ev = events_df.loc[events_df["event_id"] == event_id].iloc[0]

    if method == "gbm":
        sim = gbm_simulation_from_pre_event(close_wide, logret_wide, ticker, ev["event_date"], ESTIMATION_WINDOW_DAYS, ESTIMATION_GAP_DAYS, horizon_days, MC_N_PATHS, RANDOM_SEED)
    else:
        sim = block_bootstrap_simulation_from_pre_event(close_wide, logret_wide, ticker, ev["event_date"], estimation_window=max(120, ESTIMATION_WINDOW_DAYS), estimation_gap=ESTIMATION_GAP_DAYS, horizon_days=horizon_days, n_paths=MC_N_PATHS, block_size=5, random_seed=RANDOM_SEED)

    actual = actual_price_path(close_wide, ticker, ev["event_date"], horizon_days)
    actual_terminal = np.nan if actual.empty else actual["actual_price"].iloc[-1]
    terminal = sim.iloc[min(len(sim) - 1, max(0, len(actual) - 1))]

    if pd.isna(actual_terminal):
        location = np.nan
    elif actual_terminal > terminal["p95"]:
        location = "above_p95"
    elif actual_terminal < terminal["p05"]:
        location = "below_p05"
    elif actual_terminal > terminal["p50"]:
        location = "above_median"
    else:
        location = "below_median"

    return {
        "event_id": event_id,
        "event_name": ev["event_name"],
        "event_date": ev["event_date"],
        "ticker": ticker,
        "method": method,
        "actual_terminal_price": actual_terminal,
        "p05": terminal["p05"],
        "p50": terminal["p50"],
        "p95": terminal["p95"],
        "scenario_location": location,
    }

rows = []
for eid in events_df["event_id"]:
    for ticker in TRADE_TICKERS:
        try:
            rows.append(mc_terminal_diagnostic(eid, ticker, method="gbm"))
        except Exception as e:
            rows.append({"event_id": eid, "ticker": ticker, "method": "gbm", "error": str(e)})

mc_diagnostics = pd.DataFrame(rows)
mc_diagnostics.head(20)


## Article-ready event brief

In [ ]:

def article_brief(event_id: str, post_days: int = POST_EVENT_DAYS):
    ev = events_df.loc[events_df["event_id"] == event_id].iloc[0]
    trading_date = nearest_trading_day(ev["event_date"], close_wide.index, method="next")

    subset = summary_trade[summary_trade["event_id"] == event_id].copy()
    if subset.empty:
        print("No summary available for", event_id)
        return

    best = subset.sort_values("cum_return_0_to_post", ascending=False).iloc[0]
    worst = subset.sort_values("cum_return_0_to_post", ascending=True).iloc[0]

    ctx = (
        event_windows[
            (event_windows["event_id"] == event_id)
            & (event_windows["ticker"].isin(CONTEXT_TICKERS))
            & (event_windows["relative_day"].between(0, post_days))
        ]
        .groupby("ticker")["daily_return"]
        .apply(lambda s: (1 + s.fillna(0)).prod() - 1)
        .sort_values(ascending=False)
    )

    print("=" * 100)
    print(f"Event: {ev['event_name']}")
    print(f"Calendar date: {ev['event_date'].date()} | Trading date used: {trading_date.date()}")
    print(f"Category: {ev['category']} | Region: {ev['country_region']}")
    print()
    print("Research angle:")
    print(f"- {ev['agri_relevance']}")
    print()
    print(f"{post_days}-trading-day agri ETF reaction:")
    print(f"- Strongest: {best['ticker']} ({best['cum_return_0_to_post']:.2%})")
    print(f"- Weakest:   {worst['ticker']} ({worst['cum_return_0_to_post']:.2%})")
    print()
    print("Context ETFs:")
    for ticker, val in ctx.items():
        print(f"- {ticker}: {val:.2%}")
    print()
    print("Possible article paragraph:")
    print(
        f"Around {ev['event_name']}, the agri ETF panel showed a wide dispersion of outcomes. "
        f"{best['ticker']} had the strongest {post_days}-day move, while {worst['ticker']} lagged. "
        f"This does not prove causality, but it gives a useful market-reaction map for monitoring how "
        f"political risk, USD, crude oil, and broad commodity conditions intersect with agriculture-linked ETFs."
    )
    print("=" * 100)

article_brief(events_df["event_id"].iloc[0])



## Suggested weekly article structure

1. Event background and why it matters for agri markets.
2. Pre-event market setup: trend, volatility, USD/crude/broad commodities.
3. Event-window ETF reaction table.
4. Context comparison: DBC, UUP, USO.
5. Brownian-motion or bootstrap scenario band.
6. Interpretation: continuation, reversal, or noise.
7. Caveat: event-study correlation, not causal proof.
